In [ ]:
!uv sync
!source ../venv/bin/activate
!uv add pandas


Resolved 69 packages in 7ms
Uninstalled 26 packages in 285ms
 - appnope==0.1.4
 - asttokens==3.0.1
 - comm==0.2.3
 - debugpy==1.8.19
 - decorator==5.2.1
 - executing==2.2.1
 - ipykernel==7.1.0
 - ipython==9.8.0
 - ipython-pygments-lexers==1.1.1
 - jedi==0.19.2
 - jupyter-client==8.7.0
 - jupyter-core==5.9.1
 - matplotlib-inline==0.2.1
 - nest-asyncio==1.6.0
 - parso==0.8.5
 - pexpect==4.9.0
 - platformdirs==4.5.1
 - prompt-toolkit==3.0.52
 - ptyprocess==0.7.0
 - pure-eval==0.2.3
 - pygments==2.19.2
 - pyzmq==27.1.0
 - stack-data==0.6.3
 - tornado==6.5.4
 - traitlets==5.14.3
 - wcwidth==0.2.14
Resolved 69 packages in 4ms
Audited 64 packages in 2ms


In [34]:
import json
import glob
from pathlib import Path
import pandas as pd


In [35]:
from hmac import new


def read_json(filename):
    with open(filename, newline='') as f:
        data = json.load(f)
    return data

results_path = Path("../results/")
timestamp_dirs = [d for d in results_path.iterdir() if d.is_dir()]
json_files = list(results_path.glob("*.json"))

all_results = []

# Get the newest timestamp directory
newest_dir = None
if timestamp_dirs:
    newest_dir = sorted(timestamp_dirs, key=lambda x: x.name)[-1]
    print(f"\nNewest timestamp directory: {newest_dir.name}")
    
    print(f"\nContents of {newest_dir.name}:")
    for item in newest_dir.iterdir():
        quantum_computer = item.name
        print(f"- {quantum_computer}:")
        if item.is_dir():
            for optimising_algorithm in item.iterdir():
                print(f"-- {optimising_algorithm.name}:")
                if optimising_algorithm.is_dir():
                    for mapping_algorithm in optimising_algorithm.iterdir():
                        print(f"--- {mapping_algorithm.name}:")
                        if mapping_algorithm.is_file():
                            if mapping_algorithm.suffix == ".json":
                                data = read_json(mapping_algorithm)
                                enriched_result = {
                                    'timestamp': newest_dir.name,
                                    'topology': quantum_computer,
                                    'algorithm': optimising_algorithm.name,
                                    'optimizer': mapping_algorithm.name,
                                    'swap_count': data.get('swap_count', None),
                                    'cx_count': data.get('cx_count', None),
                                    'depth': data.get('depth', None),
                                    'optimisation_time': data.get('optimisation_time', None),
                                    'file_path': str(mapping_algorithm)
                                }
                                
                                all_results.append(enriched_result)
                                if isinstance(data, dict):
                                    print(f"    Swap Count: {data.get('swap_count', 'N/A')}, Depth: {data.get('depth', 'N/A')}, Time (ms): {data.get('optimisation_time', 'N/A')*1000 if 'optimisation_time' in data else 'N/A'}")
                                else:
                                    print(f"    Incorrect results structure")
        else:
            print(f"Incorrect results structure")
else:
    print("No timestamp directories found")



Newest timestamp directory: 1767033729

Contents of 1767033729:
- ibm_falcon_q27:
-- vqe_demo:
--- rustiq.json:
    Swap Count: 16, Depth: 111, Time (ms): 22.352218627929688
--- doustra.json:
    Swap Count: 90, Depth: 161, Time (ms): 65.27996063232422
--- pauliforest.json:
    Swap Count: 18, Depth: 43, Time (ms): 44.45195198059082
--- sabre.json:
    Swap Count: 22, Depth: 107, Time (ms): 260.42866706848145
--- qiskit.json:
    Swap Count: 23, Depth: 140, Time (ms): 23.478984832763672
--- qiskit_ai.json:
    Swap Count: 43, Depth: 311, Time (ms): 146.4848518371582
-- vqe_H2:
--- rustiq.json:
    Swap Count: 4, Depth: 42, Time (ms): 12.595176696777344
--- doustra.json:
    Swap Count: 4, Depth: 25, Time (ms): 33.486127853393555
--- pauliforest.json:
    Swap Count: 4, Depth: 28, Time (ms): 31.895160675048828
--- sabre.json:
    Swap Count: 0, Depth: 56, Time (ms): 29.500961303710938
--- qiskit.json:
    Swap Count: 0, Depth: 74, Time (ms): 23.88310432434082
--- qiskit_ai.json:
    Sw

In [36]:
df = pd.DataFrame(all_results)
print(df)


      timestamp           topology     algorithm         optimizer  \
0    1767033729     ibm_falcon_q27      vqe_demo       rustiq.json   
1    1767033729     ibm_falcon_q27      vqe_demo      doustra.json   
2    1767033729     ibm_falcon_q27      vqe_demo  pauliforest.json   
3    1767033729     ibm_falcon_q27      vqe_demo        sabre.json   
4    1767033729     ibm_falcon_q27      vqe_demo       qiskit.json   
..          ...                ...           ...               ...   
283  1767033729  ibm_rochester_q53  qaoa_max_cut      doustra.json   
284  1767033729  ibm_rochester_q53  qaoa_max_cut  pauliforest.json   
285  1767033729  ibm_rochester_q53  qaoa_max_cut        sabre.json   
286  1767033729  ibm_rochester_q53  qaoa_max_cut       qiskit.json   
287  1767033729  ibm_rochester_q53  qaoa_max_cut    qiskit_ai.json   

     swap_count  cx_count  depth  optimisation_time  \
0            16        76    111           0.022352   
1            90       303    161           0.0652

In [41]:
# Get tables according to the algorithm
if len(df) > 0:
    for algorithm in df['algorithm'].unique():
        # Filter data for this algorithm
        algorithm_df = df[df['algorithm'] == algorithm]
        
        print(f"\n Algorithm: {algorithm.upper()}")
        print("-" * 60)
        
        # Create a clean display table
        display_columns = ['topology', 'optimizer', 'swap_count', 'depth', 'optimisation_time']
        algorithm_table = algorithm_df[display_columns].copy()
        
        # Format the optimization time to show in milliseconds
        algorithm_table['optimisation_time_ms'] = algorithm_table['optimisation_time'].apply(
            lambda x: f"{x*1000:.2f}" if x is not None else "N/A"
        )
        
        # Drop the original time column and rename
        algorithm_table = algorithm_table.drop('optimisation_time', axis=1)
        algorithm_table = algorithm_table.rename(columns={
            'topology': 'Topology',
            'optimizer': 'Optimizer',
            'swap_count': 'SWAP Count',
            'cx_count': 'CNOT Count',
            'depth': 'Circuit Depth',
            'optimisation_time_ms': 'Time (ms)'
        })
        
        # Display the table
        print(algorithm_table.to_string(index=False))
        
        # Add summary statistics
        numeric_cols = algorithm_df[['swap_count', 'depth', 'optimisation_time']].select_dtypes(include=['number'])
        if not numeric_cols.empty:
            print(f"\n Summary for {algorithm}:")
            print(f"   • Average SWAP count: {algorithm_df['swap_count'].mean():.2f}")
            print(f"   • Average circuit depth: {algorithm_df['depth'].mean():.2f}")
            print(f"   • Average optimization time: {algorithm_df['optimisation_time'].mean()*1000:.2f} ms")
            
            # Find best optimizer for each metric
            if len(algorithm_df) > 1:
                best_swap = algorithm_df.loc[algorithm_df['swap_count'].idxmin()]
                best_depth = algorithm_df.loc[algorithm_df['depth'].idxmin()]
                best_time = algorithm_df.loc[algorithm_df['optimisation_time'].idxmin()]
                
                print(f"   • Best SWAP count: {best_swap['optimizer']} ({best_swap['swap_count']} swaps)")
                print(f"   • Best circuit depth: {best_depth['optimizer']} ({best_depth['depth']} gates)")
                print(f"   • Fastest optimization: {best_time['optimizer']} ({best_time['optimisation_time']*1000:.2f} ms)")
        
        print("\n" + "=" * 80)



 Algorithm: VQE_DEMO
------------------------------------------------------------
             Topology        Optimizer  SWAP Count  Circuit Depth Time (ms)
       ibm_falcon_q27      rustiq.json          16            111     22.35
       ibm_falcon_q27     doustra.json          90            161     65.28
       ibm_falcon_q27 pauliforest.json          18             43     44.45
       ibm_falcon_q27       sabre.json          22            107    260.43
       ibm_falcon_q27      qiskit.json          23            140     23.48
       ibm_falcon_q27   qiskit_ai.json          43            311    146.48
       ibm_eagle_q127      rustiq.json          22            101     30.54
       ibm_eagle_q127     doustra.json         116            161     97.00
       ibm_eagle_q127 pauliforest.json          18             40    796.38
       ibm_eagle_q127       sabre.json          23             92    383.13
       ibm_eagle_q127      qiskit.json          23            140     32.98
     

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def escape_latex(text):
    """Escape special LaTeX characters, particularly underscores."""
    return str(text).replace('_', r'\_')

def format_time_comma(time_ms):
    """Format time with comma as decimal separator."""
    return f"{time_ms:.2f}".replace('.', ',')

def generate_latex_tables(df, use_longtable=False, topologies=None):
    """
    Generate LaTeX tables for each algorithm, optimized for A4 page.
    Includes CNOT count (cx_count).
    Sorted by Topology, then Alphabetically by Optimizer.
    """
    
    latex_output = []
    
    # Filter topologies if specified
    if topologies is not None:
        df = df[df['topology'].isin(topologies)].copy()
    
    for algorithm in sorted(df['algorithm'].unique()):
        algorithm_df = df[df['algorithm'] == algorithm].copy()
        
        # --- MODIFICATION: Sort by topology AND optimizer ---
        algorithm_df = algorithm_df.sort_values(['topology', 'optimizer'])
        
        # Format algorithm name for display
        algo_display = escape_latex(algorithm.replace('_', ' ').title())
        
        # Create section
        latex_output.append(f"\n\\section*{{{algo_display}}}\n")
        
        # Prepare data
        table_data = algorithm_df[['topology', 'optimizer', 'swap_count', 'cx_count', 'depth', 'optimisation_time']].copy()
        
        # Format optimizer names
        table_data['optimizer'] = table_data['optimizer'].str.replace('.json', '', regex=False)
        
        # Convert optimization time to milliseconds and round
        table_data['optimisation_time'] = (table_data['optimisation_time'] * 1000).round(2)
        
        # Group by topology (maintains sort order from algorithm_df)
        topo_list = table_data['topology'].unique()
        
        # Start table
        if use_longtable:
            latex_output.append(r"\begin{longtable}{l|l|r|r|r|r}")
            latex_output.append(r"\caption{" + f"Results for {algo_display} algorithm" + r"} \\")
            latex_output.append(f"\\label{{tab:{algorithm}_full}} \\\\")
            latex_output.append(r"\toprule")
            latex_output.append(r"Topology & Optimizer & SWAP count & CNOT count & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
            latex_output.append(r"\endfirsthead")
            latex_output.append(r"\multicolumn{6}{c}{{\tablename\ \thetable{} -- continued from previous page}} \\")
            latex_output.append(r"\toprule")
            latex_output.append(r"Topology & Optimizer & SWAP count & CNOT count & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
            latex_output.append(r"\endhead")
            latex_output.append(r"\midrule")
            latex_output.append(r"\multicolumn{6}{r}{{Continued on next page}} \\")
            latex_output.append(r"\endfoot")
            latex_output.append(r"\bottomrule")
            latex_output.append(r"\endlastfoot")
        else:
            latex_output.append(r"\begin{table}[htbp]")
            latex_output.append(r"\centering")
            latex_output.append(r"\adjustbox{max width=\textwidth}{")
            latex_output.append(r"\begin{tabular}{l|l|r|r|r|r}")
            latex_output.append(r"\toprule")
            latex_output.append(r"Topology & Optimizer & SWAP count & CNOT count & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
        
        for i, topology in enumerate(topo_list):
            # topo_data will be sorted by optimizer because table_data was sorted
            topo_data = table_data[table_data['topology'] == topology]
            
            # Format topology name
            topo_display = escape_latex(topology.replace('_', ' ').title())
            
            for j, (_, row) in enumerate(topo_data.iterrows()):
                optimizer_escaped = escape_latex(row['optimizer'])
                
                if j == 0:
                    latex_output.append(
                        f"\\multirow{{{len(topo_data)}}}{{*}}{{{topo_display}}} & "
                        f"{optimizer_escaped} & "
                        f"{int(row['swap_count'])} & "
                        f"{int(row['cx_count'])} & "
                        f"{int(row['depth'])} & "
                        f"{format_time_comma(row['optimisation_time'])} \\\\"
                    )
                else:
                    latex_output.append(
                        f"& {optimizer_escaped} & "
                        f"{int(row['swap_count'])} & "
                        f"{int(row['cx_count'])} & "
                        f"{int(row['depth'])} & "
                        f"{format_time_comma(row['optimisation_time'])} \\\\"
                    )
            
            if i < len(topo_list) - 1:
                latex_output.append(r"\hline")
        
        # End table
        if use_longtable:
            latex_output.append(r"\end{longtable}")
        else:
            latex_output.append(r"\bottomrule")
            latex_output.append(r"\end{tabular}")
            latex_output.append(r"}")
            latex_output.append(f"\\caption{{Results for {algo_display} algorithm}}")
            latex_output.append(f"\\label{{tab:{algorithm}}}")
            latex_output.append(r"\end{table}")
        
        # Add summary statistics table
        latex_output.append("\n\\vspace{1em}\n")
        latex_output.append(r"\begin{table}[htbp]")
        latex_output.append(r"\centering")
        latex_output.append(r"\begin{tabular}{lrrrrr}")
        latex_output.append(r"\toprule")
        latex_output.append(r"Optimizer & Avg SWAP & Avg CNOT & Avg depth & Avg time (ms) & Count \\")
        latex_output.append(r"\midrule")
        
        # Ensure summary is also sorted alphabetically
        for optimizer in sorted(algorithm_df['optimizer'].str.replace('.json', '', regex=False).unique()):
            opt_data = algorithm_df[algorithm_df['optimizer'].str.replace('.json', '', regex=False) == optimizer]
            
            avg_swap = opt_data['swap_count'].mean()
            avg_cx = opt_data['cx_count'].mean()
            avg_depth = opt_data['depth'].mean()
            avg_time = opt_data['optimisation_time'].mean() * 1000
            count = len(opt_data)
            
            latex_output.append(
                f"{escape_latex(optimizer)} & "
                f"{avg_swap:.1f}".replace('.', ',') + " & "
                f"{avg_cx:.1f}".replace('.', ',') + " & "
                f"{avg_depth:.1f}".replace('.', ',') + " & "
                f"{format_time_comma(avg_time)} & "
                f"{count} \\\\"
            )
        
        latex_output.append(r"\bottomrule")
        latex_output.append(r"\end{tabular}")
        latex_output.append(f"\\caption{{Summary statistics for {algo_display} algorithm}}")
        latex_output.append(f"\\label{{tab:{algorithm}_summary}}")
        latex_output.append(r"\end{table}")
        
        latex_output.append("\n\\clearpage\n")
    
    return '\n'.join(latex_output)


def generate_compact_latex_tables(df, use_longtable=False, topologies=None):
    """
    Generate more compact LaTeX tables with abbreviated topology names.
    Sorted by Device, then Alphabetically by Optimizer.
    """
    
    latex_output = []
    
    topo_abbrev = {
        'ibm_falcon_q27': 'Falcon-27',
        'ibm_eagle_q127': 'Eagle-127',
        'ionq_harmony_q9': 'Harmony-9',
        'ibm_manhattan_q65': 'Manhattan-65',
        'ibm_reuschlikon_q16': 'Rueschlikon-16',
        'rigetti_novera_q9': 'Novera-9',
        'hexagonal_lattice_q54': 'Hex-54',
        'google_willow_q105': 'Willow-105',
        'ibm_heron_q133': 'Heron-133',
        'riken_fujitsu_q256': 'Riken-256',
        'ibm_almaden_q20': 'Almaden-20',
        'ibm_montreal_q27': 'Montreal-27',
        'ibm_cambridge_q28': 'Cambridge-28',
        'ibm_tokyo_q20': 'Tokyo-20',
        'ibm_paughkeepsie_q20': 'Poughkeepsie-20',
        'ibm_rochester_q53': 'Rochester-53'
    }
    
    if topologies is not None:
        df = df[df['topology'].isin(topologies)].copy()
    
    for algorithm in sorted(df['algorithm'].unique()):
        algorithm_df = df[df['algorithm'] == algorithm].copy()
        
        # --- MODIFICATION: Sort by topology AND optimizer ---
        algorithm_df = algorithm_df.sort_values(['topology', 'optimizer'])
        
        algo_display = escape_latex(algorithm.replace('_', ' ').upper())
        
        latex_output.append(f"\n\\section*{{{algo_display}}}\n")
        
        if use_longtable:
            latex_output.append(r"\begin{longtable}{l|l|r|r|r|r}")
            latex_output.append(r"\caption{" + f"Results for {algo_display} algorithm" + r"} \\")
            latex_output.append(f"\\label{{tab:{algorithm}_full}} \\\\")
            latex_output.append(r"\toprule")
            latex_output.append(r"Device & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
            latex_output.append(r"\endfirsthead")
            latex_output.append(r"\multicolumn{6}{c}{{\tablename\ \thetable{} -- continued from previous page}} \\")
            latex_output.append(r"\toprule")
            latex_output.append(r"Device & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
            latex_output.append(r"\endhead")
            latex_output.append(r"\midrule")
            latex_output.append(r"\multicolumn{6}{r}{{Continued on next page}} \\")
            latex_output.append(r"\endfoot")
            latex_output.append(r"\bottomrule")
            latex_output.append(r"\endlastfoot")
        else:
            latex_output.append(r"\begin{table}[htbp]")
            latex_output.append(r"\centering")
            latex_output.append(r"\small")
            latex_output.append(r"\begin{tabular}{l|l|r|r|r|r}")
            latex_output.append(r"\toprule")
            latex_output.append(r"Device & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
            latex_output.append(r"\midrule")
        
        topo_list = algorithm_df['topology'].unique()
        
        for i, topology in enumerate(topo_list):
            topo_data = algorithm_df[algorithm_df['topology'] == topology]
            topo_abbr = topo_abbrev.get(topology, escape_latex(topology))
            
            for j, (_, row) in enumerate(topo_data.iterrows()):
                optimizer = row['optimizer'].replace('.json', '').replace('pauliforest', 'pforest')
                optimizer_escaped = escape_latex(optimizer)
                
                if j == 0:
                    latex_output.append(
                        f"\\multirow{{{len(topo_data)}}}{{*}}{{{topo_abbr}}} & "
                        f"{optimizer_escaped} & "
                        f"{int(row['swap_count'])} & "
                        f"{int(row['cx_count'])} & "
                        f"{int(row['depth'])} & "
                        f"{format_time_comma(row['optimisation_time']*1000)} \\\\"
                    )
                else:
                    latex_output.append(
                        f"& {optimizer_escaped} & "
                        f"{int(row['swap_count'])} & "
                        f"{int(row['cx_count'])} & "
                        f"{int(row['depth'])} & "
                        f"{format_time_comma(row['optimisation_time']*1000)} \\\\"
                    )
            
            if i < len(topo_list) - 1:
                latex_output.append(r"\hline")
        
        if use_longtable:
            latex_output.append(r"\end{longtable}")
        else:
            latex_output.append(r"\bottomrule")
            latex_output.append(r"\end{tabular}")
            latex_output.append(f"\\caption{{Results for {algo_display} algorithm}}")
            latex_output.append(f"\\label{{tab:{algorithm}}}")
            latex_output.append(r"\end{table}")
        
        latex_output.append("\n")
    
    return '\n'.join(latex_output)


def generate_single_algorithm_table(algo_df, algorithm, use_longtable=False):
    """
    Generate a table for a single algorithm.
    Sorted by Topology, then Alphabetically by Optimizer.
    """
    latex_output = []
    
    algorithm_df = algo_df.copy()
    
    # --- MODIFICATION: Sort by topology AND optimizer ---
    algorithm_df = algorithm_df.sort_values(['topology', 'optimizer'])
    
    algo_display = escape_latex(algorithm.replace('_', ' ').title())
    
    table_data = algorithm_df[['topology', 'optimizer', 'swap_count', 'cx_count', 'depth', 'optimisation_time']].copy()
    table_data['optimizer'] = table_data['optimizer'].str.replace('.json', '', regex=False)
    table_data['optimisation_time'] = (table_data['optimisation_time'] * 1000).round(2)
    
    topo_list = table_data['topology'].unique()
    
    if use_longtable:
        latex_output.append(r"\begin{longtable}{l|l|r|r|r|r}")
        latex_output.append(r"\caption{" + f"Results for {algo_display} algorithm" + r"} \\")
        latex_output.append(f"\\label{{tab:{algorithm}_full}} \\\\")
        latex_output.append(r"\toprule")
        latex_output.append(r"Topology & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
        latex_output.append(r"\midrule")
        latex_output.append(r"\endfirsthead")
        latex_output.append(r"\multicolumn{6}{c}{{\tablename\ \thetable{} -- continued from previous page}} \\")
        latex_output.append(r"\toprule")
        latex_output.append(r"Topology & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
        latex_output.append(r"\midrule")
        latex_output.append(r"\endhead")
        latex_output.append(r"\midrule")
        latex_output.append(r"\multicolumn{6}{r}{{Continued on next page}} \\")
        latex_output.append(r"\endfoot")
        latex_output.append(r"\bottomrule")
        latex_output.append(r"\endlastfoot")
    else:
        latex_output.append(r"\begin{table}[htbp]")
        latex_output.append(r"\centering")
        latex_output.append(r"\small")
        latex_output.append(r"\begin{tabular}{l|l|r|r|r|r}")
        latex_output.append(r"\toprule")
        latex_output.append(r"Topology & Optimizer & SWAP & CNOT & Depth & Time (ms) \\")
        latex_output.append(r"\midrule")
    
    for i, topology in enumerate(topo_list):
        topo_data = table_data[table_data['topology'] == topology]
        topo_display = escape_latex(topology.replace('_', ' ').title())
        
        for j, (_, row) in enumerate(topo_data.iterrows()):
            optimizer_escaped = escape_latex(row['optimizer'])
            
            if j == 0:
                latex_output.append(
                    f"\\multirow{{{len(topo_data)}}}{{*}}{{{topo_display}}} & "
                    f"{optimizer_escaped} & "
                    f"{int(row['swap_count'])} & "
                    f"{int(row['cx_count'])} & "
                    f"{int(row['depth'])} & "
                    f"{format_time_comma(row['optimisation_time'])} \\\\"
                )
            else:
                latex_output.append(
                    f"& {optimizer_escaped} & "
                    f"{int(row['swap_count'])} & "
                    f"{int(row['cx_count'])} & "
                    f"{int(row['depth'])} & "
                    f"{format_time_comma(row['optimisation_time'])} \\\\"
                )
        
        if i < len(topo_list) - 1:
            latex_output.append(r"\hline")
    
    if use_longtable:
        latex_output.append(r"\end{longtable}")
    else:
        latex_output.append(r"\bottomrule")
        latex_output.append(r"\end{tabular}")
        latex_output.append(f"\\caption{{Results for {algo_display} algorithm}}")
        latex_output.append(f"\\label{{tab:{algorithm}}}")
        latex_output.append(r"\end{table}")
    
    return '\n'.join(latex_output)


def save_tables_individually(df, output_dir='latex_tables', use_longtable=False, topologies=None):
    """Save each algorithm's table to a separate file."""
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    if topologies is not None:
        df = df[df['topology'].isin(topologies)].copy()
        print(f"Filtering to {len(topologies)} topologies")
    
    print(f"Generating individual table files in '{output_dir}/'...")
    
    for algorithm in sorted(df['algorithm'].unique()):
        algo_df = df[df['algorithm'] == algorithm]
        table_content = generate_single_algorithm_table(algo_df, algorithm, use_longtable)
        
        filename = output_path / f"table_{algorithm}.tex"
        with open(filename, 'w') as f:
            f.write(table_content)
        
        print(f"✓ Saved: {filename}")
    
    # Create master include file
    master_content = ["% Include all algorithm tables", ""]
    for algorithm in sorted(df['algorithm'].unique()):
        master_content.append(f"\\input{{{output_dir}/table_{algorithm}.tex}}")
        master_content.append("")
    
    with open(output_path / "all_tables.tex", 'w') as f:
        f.write('\n'.join(master_content))
    
    print(f"✓ Saved master file: {output_path / 'all_tables.tex'}")


# Example usage (assuming df is already loaded)
print("="*80)
print("OPTION 1: Individual Files (RECOMMENDED for frequent updates)")
print("="*80)
save_tables_individually(df, output_dir='latex_tables', use_longtable=False)

print("\n" + "="*80)
print("OPTION 1b: Individual Files with LONGTABLE (multi-page support)")
print("="*80)
save_tables_individually(df, output_dir='latex_tables_long', use_longtable=True)

print("\n" + "="*80)
print("OPTION 1c: Subset of topologies (example)")
print("="*80)
subset_topologies = ['rigetti_novera_q9', 'ibm_eagle_q127', 'google_willow_q105', 'riken_fujitsu_q256']
save_tables_individually(df, output_dir='latex_tables_subset', 
                        use_longtable=False, topologies=subset_topologies)

print("\n" + "="*80)
print("OPTION 2: Single Combined File")
print("="*80)
full_latex = generate_latex_tables(df, use_longtable=False)
with open('all_tables_combined.tex', 'w') as f:
    f.write(full_latex)
print("✓ Saved to 'all_tables_combined.tex'")

print("\n" + "="*80)
print("OPTION 3: Compact Version with LONGTABLE")
print("="*80)
compact_latex = generate_compact_latex_tables(df, use_longtable=True)
with open('all_tables_compact_long.tex', 'w') as f:
    f.write(compact_latex)
print("✓ Saved to 'all_tables_compact_long.tex'")


OPTION 1: Individual Files (RECOMMENDED for frequent updates)
Generating individual table files in 'latex_tables/'...
✓ Saved: latex_tables/table_qaoa_max_cut.tex
✓ Saved: latex_tables/table_vqe_H2.tex
✓ Saved: latex_tables/table_vqe_demo.tex

OPTION 1b: Individual Files with LONGTABLE (multi-page support)
Generating individual table files in 'latex_tables_long/'...
✓ Saved: latex_tables_long/table_qaoa_max_cut.tex
✓ Saved: latex_tables_long/table_vqe_H2.tex
✓ Saved: latex_tables_long/table_vqe_demo.tex

OPTION 1c: Subset of topologies (example)
Filtering to 4 topologies
Generating individual table files in 'latex_tables_subset/'...
✓ Saved: latex_tables_subset/table_qaoa_max_cut.tex
✓ Saved: latex_tables_subset/table_vqe_H2.tex
✓ Saved: latex_tables_subset/table_vqe_demo.tex

OPTION 2: Single Combined File
✓ Saved to 'all_tables_combined.tex'

OPTION 3: Compact Version with LONGTABLE
✓ Saved to 'all_tables_compact_long.tex'
